# AI De-identification Demo

Demonstrates sensitive data extraction and de-identification using Snowflake Cortex LLM functions.

**Features:**
- Extract PII entities (person names, Australian phone numbers, emails, driver's licenses, credit cards)
- Validate extracted entities against specific rules
- Clean/normalize extracted values
- Tokenize values using SHA256
- Redact sensitive info in original text with tokens

## Setup: Set Database and Schema

In [ ]:
USE DATABASE CEP_PROD;
CREATE SCHEMA IF NOT EXISTS RAW;
USE SCHEMA RAW;

## Step 1: Create the LLM Entity Extraction Function

This function uses Snowflake Cortex to extract sensitive entities from text with validation rules.

In [ ]:
CREATE OR REPLACE FUNCTION LLM_EXTRACT_ENTITIES(raw_text TEXT)
RETURNS OBJECT
LANGUAGE SQL
AS
$$
SNOWFLAKE.CORTEX.TRY_COMPLETE(
    'claude-sonnet-4-5',
    [
        {
            'role': 'system',
            'content': '
            Act as a sensitive data extraction system. Your task is to analyze the provided RAW_TEXT 
            and extract all instances of the following and ONLY the following Information Types (INFO_TYPES)

            -   PERSON_NAME
            -   AUSTRALIAN_PHONE_NUMBER
            -   EMAIL_ADDRESS
            -   AUSTRALIAN_DRIVERS_LICENSE
            -   CREDIT_CARD_NUMBER

            ---
            VALIDATION RULES (only extract if valid):

            AUSTRALIAN_PHONE_NUMBER - Must satisfy ALL:
              * Contains 10 digits total (excluding country code +61)
              * Mobile numbers start with 04 (or +614)
              * Landline numbers start with 02, 03, 07, or 08 (area codes)
              * DO NOT flag: numbers with fewer than 10 digits, numbers starting with invalid prefixes, order IDs, ticket numbers, or reference codes

            CREDIT_CARD_NUMBER - Must satisfy ALL:
              * Contains exactly 13-19 digits
              * Starts with valid prefix: 4 (Visa), 5 (Mastercard), 3 (Amex/Diners), 6 (Discover)
              * DO NOT flag: numbers that are too short/long, sequential numbers like 1234567890, or obviously fake patterns like 0000-0000-0000-0000

            EMAIL_ADDRESS - Must satisfy ALL:
              * Contains exactly one @ symbol
              * Has valid format: local-part@domain.tld
              * Domain has at least one dot with valid TLD
              * DO NOT flag: partial emails, @mentions without domain, or malformed addresses like "user@" or "@domain.com"

            AUSTRALIAN_DRIVERS_LICENSE - Must match ONE of these state formats:
              * NSW (New South Wales): 6-8 alphanumeric characters (e.g., "AB123456", "1234567")
              * VIC (Victoria): 9-10 digits only (e.g., "123456789", "0123456789")
              * QLD (Queensland): 8-9 digits only (e.g., "12345678", "123456789")
              * SA (South Australia): 6 alphanumeric characters, typically starting with letter (e.g., "S12345", "AB1234")
              * WA (Western Australia): 7 digits only (e.g., "1234567")
              * TAS (Tasmania): 6-8 alphanumeric characters (e.g., "A12345", "AB123456")
              * NT (Northern Territory): 6-10 digits only (e.g., "123456", "1234567890")
              * ACT (Australian Capital Territory): 10 digits only (e.g., "1234567890")
              * May be prefixed with state code (e.g., "NSW12345678", "VIC-123456789", "QLD 12345678")
              * DO NOT flag: short numeric codes, order/reference numbers, or IDs that dont match above patterns

            ---
            OUTPUT CONSTRAINTS:
            * DO NOT EXTRACT:
              * ANY URLs (starting with http, https, ftp, ftps)
              * Invalid phone numbers, credit cards, or emails that fail validation rules above
            * If there is no sensitive information found, return empty entities array
            * IMPORTANT: The value in JSON output MUST be the exact substring from the raw_text
            * DO NOT normalize or reformat the value

            ---
            Here are some examples:

            Example 1 - RAW_TEXT: "Hi, my name is John Smith and you can reach me at john.smith@email.com or call 0412 345 678."
            Returns entities: PERSON_NAME "John Smith", EMAIL_ADDRESS "john.smith@email.com", AUSTRALIAN_PHONE_NUMBER "0412 345 678"

            Example 2 - RAW_TEXT: "The system logged error at https://api.example.com/v1/users endpoint."
            Returns: empty entities (URLs are not extracted)

            Example 3 - RAW_TEXT: "Reference number 12345, ticket ID 9876543, contact code 555-1234. Email us at support@ for help."
            Returns: empty entities (invalid data)
            '
        },
        {
            'role': 'user',
            'content': CONCAT('RAW_TEXT:\n', raw_text)
        }
    ],
    {         
        'max_tokens': 8192,
        'response_format': {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'entities': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'info_type': {
                                    'type': 'string',
                                    'enum': ['PERSON_NAME', 'AUSTRALIAN_PHONE_NUMBER', 'EMAIL_ADDRESS', 'AUSTRALIAN_DRIVERS_LICENSE', 'CREDIT_CARD_NUMBER']
                                },
                                'value': {
                                    'type': 'string'
                                }
                            },
                            'required': ['info_type', 'value']
                        }
                    }
                },
                'required': ['entities']
            }
        }
    }
)
$$;

## Step 2: Create the Clean Sensitive Value Function

Normalizes extracted values (phone numbers, emails, etc.) for consistent tokenization.

In [ ]:
CREATE OR REPLACE FUNCTION CLEAN_SENSITIVE_VALUE(i_type STRING, i_text STRING)
RETURNS STRING
LANGUAGE SQL
AS 
$$
CASE i_type
  WHEN 'AUSTRALIAN_PHONE_NUMBER'
    THEN REGEXP_REPLACE(
      REGEXP_REPLACE(i_text, '^\\+61', '0'),
      '[^0-9]', ''
    )  
  WHEN 'AUSTRALIAN_DRIVERS_LICENSE'
    THEN UPPER(REGEXP_REPLACE(i_text, '[^0-9A-Za-z]', ''))
  WHEN 'LICENSE_PLATE'
    THEN UPPER(REGEXP_REPLACE(i_text, '[^0-9A-Za-z]', ''))
  WHEN 'EMAIL_ADDRESS'
    THEN LOWER(TRIM(i_text))
  WHEN 'CREDIT_CARD_NUMBER'
    THEN REGEXP_REPLACE(i_text, '[^0-9]', '')
  WHEN 'PERSON_NAME'
    THEN UPPER(TRIM(REGEXP_REPLACE(i_text, '\\s+', ' ')))
  ELSE i_text
END
$$;

## Step 3: Create the Tokenize Value Function

A user configurable tokenisation function; SHA256 hash token for each sensitive value is used for demonstation.

> Update the tokenisation logic as per your needs.

In [ ]:
CREATE OR REPLACE FUNCTION TOKENIZE_SENSITIVE_VALUE(SENSITIVE_VALUE STRING)
RETURNS STRING
LANGUAGE SQL
AS
$$
SHA2(SENSITIVE_VALUE, 256)
$$;

## Step 4: Create the De-identify Text Function

**Main de-identification function** that orchestrates the full pipeline:

1. **Extract** - Uses LLM to identify sensitive entities (via `LLM_EXTRACT_ENTITIES`)
2. **Filter** - Removes any null/invalid extractions
3. **Clean** - Normalizes values for consistent matching (via `CLEAN_SENSITIVE_VALUE`)
4. **Tokenize** - Generates deterministic SHA256 hash tokens (via `TOKENIZE_SENSITIVE_VALUE`)
5. **Redact** - Replaces original sensitive values with tokens in the text

### Input
- `raw_text` (TEXT): The text containing potential sensitive information

### Output
Returns an OBJECT with two keys:
- `deidentified_text`: Original text with sensitive values replaced by tokens in format `__ENTITY_<TYPE>(<SHA256_TOKEN>)__`
- `extracted_entities`: Array of extracted entity objects containing: `type`, `value`, `cleaned_value`, `token`

### Example
```
Input:  'Contact John at john@email.com'
Output: {
  "deidentified_text": "Contact __ENTITY_PERSON_NAME(abc123...)__ at __ENTITY_EMAIL_ADDRESS(def456...)__",
  "extracted_entities": [
    {"type": "PERSON_NAME", "value": "John", "cleaned_value": "JOHN", "token": "abc123..."},
    {"type": "EMAIL_ADDRESS", "value": "john@email.com", "cleaned_value": "john@email.com", "token": "def456..."}
  ]
}
```

In [ ]:
-- Processing the raw_text is done in 5 steps,
-- look at description of each step to understand it's working.
CREATE OR REPLACE FUNCTION DEIDENTIFY_TEXT(raw_text TEXT)
RETURNS OBJECT
LANGUAGE SQL
AS
$$
-- Step 5: Build final output object with redacted text and entity metadata
SELECT OBJECT_CONSTRUCT(
    'deidentified_text',
    -- Step 5a: REDUCE iterates through entities, replacing each sensitive value
    -- with its tokenized placeholder in the accumulated text
    REDUCE(
        cleaned_entities,
        raw_text,
        (acc, el) -> REPLACE(
            acc,
            el:value::STRING,
            '__ENTITY_' || el:type::STRING || '(' || el:token::STRING || ')__'
        )
    ),
    'extracted_entities', cleaned_entities
)
FROM (
    -- Step 4: Add SHA256 token to each entity using the cleaned value
    SELECT TRANSFORM(
        -- Step 3: Clean/normalize values and restructure entity objects
        TRANSFORM(
            -- Step 2: Filter out any null or invalid extractions
            FILTER(
                -- Step 1: Extract entities from text using LLM
                LLM_EXTRACT_ENTITIES(raw_text):structured_output[0]:raw_message:entities,
                e -> e:value IS NOT NULL AND e:info_type IS NOT NULL
            ),
            -- Step 3a: Build cleaned entity object with normalized value
            e -> OBJECT_CONSTRUCT(
                'type', e:info_type::STRING,
                'value', e:value::STRING,
                'cleaned_value', CLEAN_SENSITIVE_VALUE(e:info_type::STRING, e:value::STRING)
            )
        ),
        -- Step 4a: Append token field using already-computed cleaned_value
        e -> OBJECT_INSERT(e, 'token', TOKENIZE_SENSITIVE_VALUE(e:cleaned_value::STRING))
    ) AS cleaned_entities
)
$$;

## Step 5: Create Sample Data

Test cases covering valid PII, invalid data, and edge cases.

In [ ]:
CREATE OR REPLACE TABLE SAMPLE_TEXT_DATA (
    id INT,
    description VARCHAR,
    raw_text VARCHAR
);

INSERT INTO SAMPLE_TEXT_DATA VALUES
(1, 'Valid PII - basic', 'Customer John Smith called regarding his account. Contact: 0412 345 678, email john.smith@gmail.com. License: NSW12345678.'),
(2, 'Valid PII - credit card', 'Sarah O''Connor submitted a complaint. Credit card 4532-1234-5678-9012 was charged incorrectly. Reach her at sarah.oconnor@company.com.au or (02) 9876 5432.'),
(3, 'URLs only - should be empty', 'Technical issue reported. Check logs at https://api.internal.com/errors and documentation at http://docs.example.org/troubleshoot.'),
(4, 'Reference numbers only - should be empty', 'Order #ORD-98765 processed. Reference: 12345. Ticket ID: TKT-5678. No customer details provided.'),
(5, 'Valid PII - landline', 'Wei Chen needs callback at 0398 765 432. Driver license VIC-87654321 verified. Email: wei.chen@outlook.com'),
(6, 'Invalid data - should be empty', 'Invalid data test: phone 123-456, partial email test@, card 1234-5678, @twitter mention.'),
(7, 'Mixed valid and invalid', 'Mixed content: Valid mobile 0478 123 456 but invalid 555-1234. Real email support@domain.com but fake user@ incomplete.'),
(8, 'Valid PII - Amex card', 'Amex card 378282246310005 used by Michael Johnson. Backup contact: 0412-987-654 or m.johnson@email.net'),
(9, 'Fake patterns - should be empty', 'Test patterns to ignore: 0000000000, 0000-0000-0000-0000, fake@fake, placeholder@test'),
(10, 'Valid PII - complete record', 'Maria Garcia, license QLD-11223344, paid with Visa 4111111111111111. Contact: (07) 3456 7890, maria.garcia@yahoo.com.au'),
(11, 'System message - should be empty', 'System notification: Server restarted at 10:45:23. Memory usage 8192MB. CPU load 45%. No PII here.'),
(12, 'Valid PII - Mastercard', 'David Lee inquired about policy. Mastercard 5425233430109903 on file. Mobile: 0423 456 789. Email david.lee@proton.me'),
(13, 'Valid PII - multiple cards', 'Callback requested by Emma Wilson at 0487-654-321. License SA-99887766. CC ending 9012 (full: 4000056655665556).'),
(14, 'Business references - should be empty', 'Reference numbers only: Case #2024-001, Invoice INV-789012, Account ACC-345678. Visit help.example.com for support.'),
(15, 'Valid PII - international format', 'International format test: +61 412 345 678 is valid Australian. Contact: james.brown@corporation.com, license TAS-55443322.');

## Step 6: Run the Demo - De-identify Sample Data

Process all sample records and display de-identified results.

In [ ]:
SELECT 
    id,
    description,
    raw_text,
    DEIDENTIFY_TEXT(raw_text):deidentified_text::STRING as deidentified_text
FROM SAMPLE_TEXT_DATA
ORDER BY id;

## Step 7: Detailed Entity Analysis

Flatten extracted entities to see individual entity details (type, original value, cleaned value, token).

In [ ]:
SELECT 
    s.id,
    s.description,
    e.value:type::STRING as entity_type,
    e.value:value::STRING as original_value,
    e.value:cleaned_value::STRING as cleaned_value,
    SUBSTR(e.value:token::STRING, 1, 16) || '...' as token_preview
FROM SAMPLE_TEXT_DATA s,
LATERAL FLATTEN(input => DEIDENTIFY_TEXT(s.raw_text):extracted_entities) e
ORDER BY s.id, entity_type;

## Step 8: Single Text Example

Test the function with a single text input.

In [ ]:
SELECT DEIDENTIFY_TEXT('Please contact Alice Johnson at alice.johnson@email.com or call 0412 555 999 regarding invoice #12345.');